In [1]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score , classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
TRAIN_PATH = "./../data/processed/train.csv"
VALIDATION_PATH = "./../data/processed/validation.csv"
TEST_PATH = "./../data/processed/test.csv"

In [3]:
train_df= pd.read_csv(TRAIN_PATH)
validation_df= pd.read_csv(VALIDATION_PATH)
test_df= pd.read_csv(TEST_PATH)

X_train = train_df.drop(columns=["label"])
y_train = train_df["label"]

X_validation = validation_df.drop(columns=["label"])
y_validation = validation_df["label"]

X_test = test_df.drop(columns=["label"])
y_test = test_df["label"]

del train_df, validation_df, test_df

In [ ]:
X_train_ids = X_train[["turn_id", "conv_id"]]
X_validation_ids = X_validation[["turn_id", "conv_id"]]
X_test_ids = X_test[["turn_id", "conv_id"]]

X_train = X_train.drop(columns=["turn_id", "conv_id"])
X_validation = X_validation.drop(columns=["turn_id", "conv_id"])
X_test = X_test.drop(columns=["turn_id", "conv_id"])

In [5]:
print("Training set shape:", X_train.shape)
print("Validation set shape:", X_validation.shape)
print("Test set shape:", X_test.shape)

Training set shape: (1952, 13)
Validation set shape: (646, 13)
Test set shape: (657, 13)


In [6]:
def save_confusion_matrix(cm, filename,title):
    plt.figure()
    sns.heatmap(cm, annot=True, fmt="d") 
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.savefig(filename)

    plt.close()

In [7]:
def evaluate_model(y_pred, y_true):
    acc = accuracy_score(y_true, y_pred)
    report = classification_report(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred)
    return acc, report, cm

In [ ]:
import pandas as pd


def group_by_conv(X, y_true, y_pred):
 
    df = X.copy()
    df["_label"] = y_true
    df["_pred"]  = y_pred

    df = df.sort_values(["conv_id"]).reset_index(drop=True)

    conv_ids       = []
    y_true_grouped = []
    y_pred_grouped = []

    for cid, group in df.groupby("conv_id", sort=False):
        conv_ids.append(cid)
        y_true_grouped.append(group["_label"].tolist())
        y_pred_grouped.append(group["_pred"].tolist())

    return conv_ids, y_true_grouped, y_pred_grouped


In [9]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(
    n_neighbors=min(10, len(X_train)-1),
    contamination=.2,
    metric="cosine"
)

y_pred_train = lof.fit_predict(X_train, y_train)
y_pred_train = [1 if p == -1 else 0 for p in y_pred_train]
acc_train, report_train, cm_train = evaluate_model(y_pred_train, y_train)



In [10]:
print ("LOF Train Accuracy:", acc_train)
print ("LOF Train Classification Report:\n", report_train)

LOF Train Accuracy: 0.6183401639344263
LOF Train Classification Report:
               precision    recall  f1-score   support

           0       0.71      0.79      0.75      1414
           1       0.24      0.17      0.20       538

    accuracy                           0.62      1952
   macro avg       0.47      0.48      0.47      1952
weighted avg       0.58      0.62      0.60      1952



In [11]:

params = {
    'penalty': 'l2',         
    'C': 1.0,                
    'solver': 'lbfgs',       
    'max_iter': 10000,        
    'random_state': 42
}

logreg = LogisticRegression(**params)
logreg.fit(X_train, y_train)

y_train_pred = logreg.predict(X_train)
y_validation_pred = logreg.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)


test_acc, test_report, test_cm = evaluate_model(logreg.predict(X_test), y_test)


c:\Users\abdo\anaconda3\envs\myenv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


Training Accuracy: 0.725922131147541
Validation Accuracy: 0.7337461300309598
Training Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.84      1414
           1       0.64      0.01      0.03       538

    accuracy                           0.73      1952
   macro avg       0.68      0.51      0.43      1952
weighted avg       0.70      0.73      0.62      1952

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.85       473
           1       1.00      0.01      0.01       173

    accuracy                           0.73       646
   macro avg       0.87      0.50      0.43       646
weighted avg       0.80      0.73      0.62       646



In [14]:
params = {
    'max_depth': 10,           
    'min_samples_split': 10,    
    'min_samples_leaf': 5,     
    'max_features': None,      
    'random_state': 42
}

dtc = DecisionTreeClassifier(**params)
dtc.fit(X_train, y_train)

y_train_pred = dtc.predict(X_train)
y_validation_pred = dtc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


test_acc, test_report, test_cm = evaluate_model(dtc.predict(X_test), y_test)

In [15]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)


Training Accuracy: 0.8555327868852459
Validation Accuracy: 0.7058823529411765
Training Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.94      0.90      1414
           1       0.79      0.64      0.71       538

    accuracy                           0.86      1952
   macro avg       0.83      0.79      0.81      1952
weighted avg       0.85      0.86      0.85      1952

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.83      0.81       473
           1       0.44      0.36      0.39       173

    accuracy                           0.71       646
   macro avg       0.61      0.60      0.60       646
weighted avg       0.69      0.71      0.70       646

Test Accuracy: 0.7397260273972602
Test Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.87      0.83       471
           1       0.56     

In [17]:
params = {
'n_estimators': 300,          
'max_depth': 7,             
'min_samples_split': 5,      
'min_samples_leaf': 5,        
'max_features': 'sqrt',       
'bootstrap': True,
'random_state': 42,
'n_jobs': -1                 
}

rfc = RandomForestClassifier(**params)
rfc.fit(X_train, y_train)
y_train_pred = rfc.predict(X_train)
y_validation_pred = rfc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)


test_acc, test_report, test_cm = evaluate_model(rfc.predict(X_test), y_test)

In [18]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)  
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)  

Training Accuracy: 0.8442622950819673
Validation Accuracy: 0.7383900928792569
Training Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.98      0.90      1414
           1       0.90      0.49      0.63       538

    accuracy                           0.84      1952
   macro avg       0.87      0.73      0.77      1952
weighted avg       0.85      0.84      0.83      1952

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.93      0.84       473
           1       0.53      0.23      0.32       173

    accuracy                           0.74       646
   macro avg       0.65      0.58      0.58       646
weighted avg       0.70      0.74      0.70       646

Test Accuracy: 0.726027397260274
Test Classification Report:
               precision    recall  f1-score   support

           0       0.75      0.92      0.83       471
           1       0.54      

In [19]:
params = {
    'n_estimators': 200,        
    'max_depth': 5,             
    'learning_rate': 0.01,       
    'random_state': 42,
    'n_jobs': -1,
}

xgb = XGBClassifier(**params)
xgb.fit(X_train, y_train)

# Predictions
y_train_pred = xgb.predict(X_train)
y_validation_pred = xgb.predict(X_validation)

# Evaluation
train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)
test_acc, test_report, test_cm = evaluate_model(xgb.predict(X_test), y_test)

In [20]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.8278688524590164
Validation Accuracy: 0.7291021671826625
Training Classification Report:
               precision    recall  f1-score   support

           0       0.82      0.98      0.89      1414
           1       0.89      0.43      0.58       538

    accuracy                           0.83      1952
   macro avg       0.85      0.71      0.74      1952
weighted avg       0.84      0.83      0.81      1952

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.92      0.83       473
           1       0.49      0.20      0.28       173

    accuracy                           0.73       646
   macro avg       0.62      0.56      0.56       646
weighted avg       0.69      0.73      0.68       646

Test Accuracy: 0.7138508371385084
Test Classification Report:
               precision    recall  f1-score   support

           0       0.74      0.92      0.82       471
           1       0.49     

In [21]:
params = {
    'C': 1.0,                
    'kernel': 'rbf',        
    'random_state': 42
}

svc = SVC(**params)
svc.fit(X_train, y_train)

y_train_pred = svc.predict(X_train)
y_validation_pred = svc.predict(X_validation)

train_acc, train_report, train_cm = evaluate_model(y_train_pred, y_train)
val_acc, val_report, val_cm = evaluate_model(y_validation_pred, y_validation)

test_acc, test_report, test_cm = evaluate_model(svc.predict(X_test), y_test)



In [22]:
print("Training Accuracy:", train_acc)
print("Validation Accuracy:", val_acc)
print("Training Classification Report:\n", train_report)
print("Validation Classification Report:\n", val_report)
print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)

Training Accuracy: 0.7346311475409836
Validation Accuracy: 0.7306501547987616
Training Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.85      1414
           1       0.95      0.04      0.07       538

    accuracy                           0.73      1952
   macro avg       0.84      0.52      0.46      1952
weighted avg       0.79      0.73      0.63      1952

Validation Classification Report:
               precision    recall  f1-score   support

           0       0.73      0.99      0.84       473
           1       0.40      0.01      0.02       173

    accuracy                           0.73       646
   macro avg       0.57      0.50      0.43       646
weighted avg       0.64      0.73      0.62       646

Test Accuracy: 0.7138508371385084
Test Classification Report:
               precision    recall  f1-score   support

           0       0.72      1.00      0.83       471
           1       0.00     

In [23]:
# fine tune on xgboost

grid_search_params = {
    'n_estimators': [100, 200, 300, 500],    
    'max_depth': [3, 5, 7,10],
    'learning_rate': [0.01, 0.1, 0.2, 0.005],
    'random_state': [42],   
    'n_jobs': [-1]
}

xgb = XGBClassifier()
grid_search = GridSearchCV(estimator=xgb, param_grid=grid_search_params, cv=3, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)   

print("Best Hyperparameters:", grid_search.best_params_)


val_acc, val_report, val_cm = evaluate_model(grid_search.predict(X_validation), y_validation)
test_acc, test_report, test_cm = evaluate_model(grid_search.predict(X_test), y_test)
print("Validation Accuracy:", val_acc)
print("Validation Classification Report:\n", val_report)

print("Test Accuracy:", test_acc)
print("Test Classification Report:\n", test_report)






Fitting 3 folds for each of 64 candidates, totalling 192 fits
Best Hyperparameters: {'learning_rate': 0.005, 'max_depth': 7, 'n_estimators': 100, 'n_jobs': -1, 'random_state': 42}
Validation Accuracy: 0.7321981424148607
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.73      1.00      0.85       473
           1       0.00      0.00      0.00       173

    accuracy                           0.73       646
   macro avg       0.37      0.50      0.42       646
weighted avg       0.54      0.73      0.62       646

Test Accuracy: 0.7168949771689498
Test Classification Report:
               precision    recall  f1-score   support

           0       0.72      1.00      0.84       471
           1       0.00      0.00      0.00       186

    accuracy                           0.72       657
   macro avg       0.36      0.50      0.42       657
weighted avg       0.51      0.72      0.60       657



c:\Users\abdo\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abdo\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\abdo\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0]